Goal: Preprocess the data, including training sentences, tokenizing and aligning labels

In [11]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification
from TorchCRF import CRF
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [12]:
label_map = {
    "O": 0,
    "ENTITY": 1, #主语
    "TIME": 5, #year
    "UNIT": 10,
    "NUMBER": 11, #number only
}

In [13]:
# Example training data
train_sentences = [
    "Amazon's net profit was $769.6 million in 2021.",
    "Microsoft's revenue rose to $5 billion in 2022.",
    "Profit in 2023 was $1.10 billion."
]

# Entity spans (from your simplified NER, e.g., ENTITY, NUMBER, TIME)
train_entities = [
    [("Amazon", "ENTITY", 0), ("net profit", "ENTITY", 2), ("$769.6 million", "NUMBER", 5), ("2021", "TIME", 8)],
    [("Microsoft", "ENTITY", 0), ("revenue", "ENTITY", 2), ("$5 billion", "NUMBER", 5), ("2022", "TIME", 8)],
    [("Profit", "ENTITY", 0), ("2023", "TIME", 2), ("$1.10 billion", "NUMBER", 4)]
]

# Labeled relationships: (entity1_idx, entity2_idx, relation)
train_relations = [
    [(0, 2, "owned_by"), (2, 5, "has_value"), (5, 8, "in_year")],
    [(0, 2, "owned_by"), (2, 5, "has_value"), (5, 8, "in_year")],
    [(0, 4, "has_value"), (4, 2, "in_year")]
]

In [14]:
import torch
from transformers import BertTokenizerFast, BertModel
from torch import nn

class BertRelationExtractor(nn.Module):
    def __init__(self, num_labels, dropout=0.1):
        super(BertRelationExtractor, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size * 2, num_labels)  # Concatenate two entity embeddings

    def forward(self, input_ids, attention_mask, entity1_mask, entity2_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Extract embeddings for entity1 and entity2 using their masks
        entity1_embeds = (hidden_states * entity1_mask.unsqueeze(-1)).sum(dim=1)  # Sum over entity1 tokens
        entity2_embeds = (hidden_states * entity2_mask.unsqueeze(-1)).sum(dim=1)  # Sum over entity2 tokens
        
        # Normalize by number of tokens (optional, to handle multi-token entities)
        entity1_embeds = entity1_embeds / entity1_mask.sum(dim=1, keepdim=True).clamp(min=1)
        entity2_embeds = entity2_embeds / entity2_mask.sum(dim=1, keepdim=True).clamp(min=1)

        # Concatenate embeddings
        combined_embeds = torch.cat([entity1_embeds, entity2_embeds], dim=-1)
        combined_embeds = self.dropout(combined_embeds)
        
        logits = self.classifier(combined_embeds)  # [batch_size, num_labels]

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            return loss
        return logits

In [15]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
relation_map = {"no_relation": 0, "has_value": 1, "in_year": 2, "owned_by": 3}
max_length = 100

In [18]:
def preprocess_relations(sentences, entities, relations):
    inputs = {"input_ids": [], "attention_mask": [], "entity1_mask": [], "entity2_mask": [], "labels": []}
    
    for sent, ent_list, rel_list in zip(sentences, entities, relations):
        tokens = sent.split()
        encoding = tokenizer(sent, return_tensors="pt", padding="max_length", max_length=max_length, truncation=True)
        input_ids = encoding["input_ids"][0]
        attention_mask = encoding["attention_mask"][0]

        # Generate all possible entity pairs
        for i, (e1_text, e1_label, e1_idx) in enumerate(ent_list):
            for j, (e2_text, e2_label, e2_idx) in enumerate(ent_list):
                if i == j:
                    continue
                
                # Create entity masks
                e1_mask = torch.zeros(max_length)
                e2_mask = torch.zeros(max_length)
                
                # Approximate token positions
                e1_start = len(tokenizer.tokenize(" ".join(tokens[:e1_idx]))) + 1  # +1 for [CLS]
                e2_start = len(tokenizer.tokenize(" ".join(tokens[:e2_idx]))) + 1
                e1_len = len(tokenizer.tokenize(e1_text))
                e2_len = len(tokenizer.tokenize(e2_text))

                e1_mask[e1_start:e1_start + e1_len] = 1
                e2_mask[e2_start:e2_start + e2_len] = 1

                # Label: positive relation or no_relation
                rel_label = next((relation_map[rel] for (idx1, idx2, rel) in rel_list if idx1 == e1_idx and idx2 == e2_idx), relation_map["no_relation"])
                
                # Convert rel_label to a tensor
                rel_label_tensor = torch.tensor(rel_label, dtype=torch.long)

                inputs["input_ids"].append(input_ids)
                inputs["attention_mask"].append(attention_mask)
                inputs["entity1_mask"].append(e1_mask)
                inputs["entity2_mask"].append(e2_mask)
                inputs["labels"].append(rel_label_tensor)
    
    return {k: torch.stack(v) for k, v in inputs.items()}

In [19]:
train_data = preprocess_relations(train_sentences, train_entities, train_relations)

In [20]:
from torch.utils.data import DataLoader, Dataset

class RelationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data["input_ids"])

    def __getitem__(self, idx):
        return {key: self.data[key][idx] for key in self.data}

In [21]:
train_dataset = RelationDataset(train_data)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertRelationExtractor(num_labels=len(relation_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [22]:
def train_model(model, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            e1_mask = batch["entity1_mask"].to(device)
            e2_mask = batch["entity2_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            loss = model(input_ids, attention_mask, e1_mask, e2_mask, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

In [23]:
train_model(model, train_loader)

Epoch 1, Loss: 1.4377
Epoch 2, Loss: 0.8303
Epoch 3, Loss: 0.7330
Epoch 4, Loss: 0.7583
Epoch 5, Loss: 0.7092
